# 🌾 GramSetu AI - 10-Lakh Persona LoRA Fine-Tuner
### Fast Fine-Tuning for Qwen 2.5 3B / Llama 3 on Free Google Colab GPU (T4/A100)
This notebook fine-tunes a base model on the 10-lakh Indian multi-dialect, multi-parameter GramSetu dataset using Unsloth & QLoRA.

In [ ]:
# 1. Install Unsloth & Training Dependencies
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# 2. Load Base Model (Qwen 2.5 3B Instruct)
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None # Auto detect (Float16 for T4, Bfloat16 for Ampere)
load_in_4bit = True # 4bit quantization to fit in 6GB VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 3. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

In [ ]:
# 4. Load & Format GramSetu Dataset
from datasets import load_dataset

# Upload your train_10lakh_personas.jsonl or load directly
dataset = load_dataset("json", data_files="train_10lakh_personas.jsonl", split="train")

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_ctx, output in zip(instructions, inputs, outputs):
        sys = "You are GramSetu AI, an expert agricultural, rural, and multidisciplinary intelligence assistant."
        user_p = f"{instruction}\n({input_ctx})" if input_ctx else instruction
        text = f"<|im_start|>system\n{sys}<|im_end|>\n<|im_start|>user\n{user_p}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

In [ ]:
# 5. Train with SFTTrainer (Takes ~10-15 mins on free Colab T4 GPU)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 150,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer.train()

In [ ]:
# 6. Export to GGUF (Ready for Ollama / GramSetu)
# Save 4-bit, 8-bit or Q4_K_M GGUF format
model.save_pretrained_gguf("gramsetu-trained-q4", tokenizer, quantization_method = "q4_k_m")

print("🎉 Model trained and saved to gramsetu-trained-q4.gguf!")
print("Download this file and use it in Ollama with `ollama create gramsetu-ai -f Modelfile`")